In [1]:
import os
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from IPython.display import IFrame

# 1. Klasör Ayarları
NOTEBOOK_DIR = os.getcwd()
PROJECT_DIR = os.path.dirname(NOTEBOOK_DIR)
RESULTS_DIR = os.path.join(PROJECT_DIR, "results")
DATA_DIR = os.path.join(PROJECT_DIR, "data")

# 2. Verileri Yükle
df_slsqp = pd.read_csv(os.path.join(RESULTS_DIR, "efficient_frontier_slsqp.csv"))
df_nsga2 = pd.read_csv(os.path.join(RESULTS_DIR, "efficient_frontier_nsga2.csv"))
df_sa = pd.read_csv(os.path.join(RESULTS_DIR, "best_solution_sa.csv"))
df_coords = pd.read_csv(os.path.join(DATA_DIR, "tr_provinces_coords.csv"))

def normalize(text):
    return text.translate(str.maketrans("çğıöşüÇĞİÖŞÜ", "cgiosucgiosu")).lower().strip()

df_coords['norm_prov'] = df_coords['province'].apply(normalize)
df_coords = df_coords.set_index('norm_prov')

# Sütunları Ayıkla
meta_cols = ['lambda', 'total_cost', 'total_risk', 'expected_production']
asset_cols = [c for c in df_slsqp.columns if c not in meta_cols]

In [7]:
# 3. Sütun isimlerindeki lon/lng varyasyonlarını otomatik yakalayalım
lon_col = [c for c in df_coords.columns if 'lon' in c.lower() or 'lng' in c.lower()][0]
lat_col = [c for c in df_coords.columns if 'lat' in c.lower()][0]

# Yan Yana Subplot Düzeni Oluştur (Sol: Frontier, Sağ: Türkiye Haritası)
fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.4, 0.6],
    subplot_titles=('Yatırımın Risk/Maliyet Dengesi', 'Türkiye Yatırım Coğrafyası'),
    specs=[[{"type": "scatter"}, {"type": "scattergeo"}]] # Sağ taraf coğrafi harita modunda
)

# --- ARKA PLAN (Statik Veriler) ---
# SLSQP Eğrisi (Mavi çizgi)
fig.add_trace(go.Scatter(
    x=df_plot['total_risk'], y=df_plot['total_cost'],
    mode='lines', name='Teorik Limit (SLSQP)',
    line=dict(color='#1f77b4', width=2, dash='dot'),
    opacity=0.5
), row=1, col=1)

# --- DİNAMİK FRAMES (Slider Hareketleri) ---
frames = []
for i in range(len(df_plot)):
    row = df_plot.iloc[i]
    budget_label = f"{row['total_cost']:,.0f} Milyon $"
    
    # A: Sol Grafik - Hareketli Gösterge
    curr_selection = go.Scatter(
        x=[row['total_risk']], y=[row['total_cost']],
        mode='markers+text', name='Seçili Bütçe',
        marker=dict(color='red', size=12, line=dict(width=2, color='white')),
        text=[f"Bütçe: {budget_label}"], textposition="top right"
    )
    
    # B: Sağ Harita - İl Bazlı Balonlar
    map_lons, map_lats, map_sizes, map_colors, map_texts = [], [], [], [], []
    for col in asset_cols:
        mw = row[col]
        if mw > 10.0:
            prov = col.split('_')[0]
            tech = col.split('_')[1]
            p_norm = normalize(prov)
            if p_norm in df_coords.index:
                map_lons.append(df_coords.loc[p_norm, lon_col])
                map_lats.append(df_coords.loc[p_norm, lat_col])
                map_sizes.append(np.sqrt(mw) * 0.6) 
                # Güneş: Turuncu, Rüzgar: Mavi
                map_colors.append('#ff7f0e' if tech == 'solar' else '#1f77b4')
                map_texts.append(f"<b>{prov.upper()}</b><br>{tech.upper()}: {mw:.1f} MW")

    trace_map = go.Scattergeo(
        lon=map_lons, lat=map_lats,
        mode='markers',
        marker=dict(size=map_sizes, color=map_colors, opacity=0.8, line=dict(width=1, color='white')),
        text=map_texts, hoverinfo='text'
    )
    
    frames.append(go.Frame(data=[curr_selection, trace_map], name=str(i), traces=[1, 2]))

# İlk görünüm
fig.add_trace(frames[0].data[0], row=1, col=1)
fig.add_trace(frames[0].data[1], row=1, col=2)
fig.frames = frames

# --- SLIDER AYARLARI (Bütçe Bazlı) ---
steps = []
for i in range(len(df_plot)):
    cost = df_plot.iloc[i]['total_cost']
    steps.append(dict(
        method="animate", label=f"{int(cost)} M$",
        args=[[str(i)], dict(mode="immediate", transition=dict(duration=0), frame=dict(duration=0, redraw=True))]
    ))

# 3. Görsel Makyaj ve Harita Sınırları
fig.update_geos(
    scope='europe',
    resolution=50,
    showcoastlines=True, coastlinecolor="RebeccaPurple",
    showland=True, landcolor="LightGreen",
    showocean=True, oceancolor="LightBlue",
    showlakes=True, lakecolor="LightBlue", # Hata Düzeltildi: oceancolor yerine lakecolor yapıldı
    showcountries=True,
    lonaxis_range=[24, 46],
    lataxis_range=[35, 43],
    projection_type="mercator"
)

fig.update_layout(
    title='<b>Karar Destek Sistemi: Bütçeye Göre Yenilenebilir Portföy Planlaması</b>',
    width=1100, height=600,
    template='plotly_white',
    sliders=[dict(
        active=0, currentvalue={"prefix": "Hedeflenen Toplam Bütçe: ", "suffix": " Milyon $"},
        pad={"t": 50}, steps=steps
    )],
    xaxis1=dict(title="Portföy Riski (Düşük Risk <---> Yüksek Risk)", showgrid=False),
    yaxis1=dict(title="Toplam Maliyet (Milyon $)")
)

# Dosyayı kaydet ve IFrame ile göster
output_path = "final_budget_dashboard.html"
pio.write_html(fig, file=output_path, auto_open=False, include_plotlyjs='cdn')
IFrame(src=output_path, width=1150, height=650)